## Property modeling

This notebook shows how to populate a structural geological model with properties (e.g. porosity) using geostatistical methods.

We use the `property_estimation` module of [gempy_plugins](https://github.com/gempy-project/gempy_plugins/tree/main/gempy_plugins/property_estimation). It wraps [GSTools](https://geostat-framework.readthedocs.io/projects/gstools/en/stable/) (kriging and random field simulation) and takes care of the bookkeeping:

- It splits the model into **domains**: one domain per combination of lithology and fault block. Property values are estimated separately in each domain, so they never "leak" across a lithological boundary or across a fault.
- You can use a different variogram (and kriging method) for every domain.
- Results are written back onto the regular grid of the GemPy model, ready for plotting.

**Workflow in this notebook:**
1. Build a structural model with GemPy
2. Look at the domains
3. Create a synthetic "true" property field with an unconditioned simulation
4. Sample the true field along virtual boreholes
5. Reconstruct the field from the borehole data with kriging
6. Generate conditioned simulations from the same borehole data
7. Explore the results interactively

In addition to GemPy, it requires [PyVista](https://pyvista.org/) and [GSTools](https://geostat-framework.readthedocs.io/projects/gstools/en/stable/). The 3D plots open in separate windows, so this notebook is meant to run locally.

In [ ]:
# Importing necessary libraries
import numpy as np
import matplotlib.pyplot as plt
import gstools as gs
from scipy.spatial import cKDTree

import gempy as gp
import gempy_viewer as gpv

# Property estimation plugin
from gempy_plugins.property_estimation.domains import compute_domains, describe_domains
from gempy_plugins.property_estimation.conditioning_data import ConditioningData
from gempy_plugins.property_estimation.kriging import KrigingDomainConfig, PropertyField, run_kriging
from gempy_plugins.property_estimation.simulation import SimulationDomainConfig, run_simulation
from gempy_plugins.property_estimation.plotting import (
    plot_domains, plot_fault_blocks, plot_conditioning_data,
    plot_property_field, plot_property_field_interactive,
)

### 1) Structural geological model

The same model as in the step-by-step example: three layers and a basement, offset by a fault. We use the full resolution of 125 x 50 x 50 cells. Computing the model takes about a minute.

In [ ]:
# Generate the model
data_path = 'https://raw.githubusercontent.com/cgre-aachen/gempy_data/master/'
path_to_data = data_path + "/data/input_data/jan_models/"

# Create a GeoModel instance
geo_model = gp.create_geomodel(
    project_name='combination',
    extent=[0, 2500, 0, 1000, 0, 1000],
    resolution=[125, 50, 50],
    importer_helper=gp.data.ImporterHelper(
        path_to_orientations=path_to_data + "model7_orientations.csv",
        path_to_surface_points=path_to_data + "model7_surface_points.csv"
    )
)

# Map geological series to surfaces
gp.map_stack_to_surfaces(
    gempy_model=geo_model,
    mapping_object={
        "Fault_Series": ('fault'),
        "Strat_Series1": ('rock3'),
        "Strat_Series2": ('rock2', 'rock1'),
    }
)
# Define youngest structural group as fault
gp.set_is_fault(geo_model, ["Fault_Series"])

# Compute the geological model
gp.compute_model(geo_model)

In [ ]:
gpv.plot_3d(geo_model, show_block=True, show_lith=True, show_data=False)

### 2) Domains

A domain is a combination of a lithology id and a fault block id, written as `(lith_id, fault_block_id)`. The same lithology on both sides of the fault becomes two separate domains.

`compute_domains` returns the lithology and fault block of every grid cell, plus the list of domains present in the model.

In [ ]:
# Lithology and fault block per cell, and the list of domains
lith_array, fault_array, domain_keys = compute_domains(geo_model)

# Human-readable names, e.g. (2, 0) -> "rock3 (fault block 0)"
domain_names = describe_domains(geo_model, domain_keys)

for key in domain_keys:
    n_cells = np.sum((lith_array == key[0]) & (fault_array == key[1]))
    print(f"{str(key):8} {domain_names[key]:28} {n_cells:7d} cells")

In [ ]:
# 3D view of all domains: each lithology has its own color, fault blocks are different shades of it
plot_domains(geo_model, lith_array, fault_array)

In [ ]:
# 3D view of the fault blocks alone
plot_fault_blocks(geo_model, fault_array)

### 3) Unconditioned simulation: a synthetic "true" property field

We don't have real measurements for this model, so we first create a synthetic "true" property field. Later we sample it along boreholes and try to reconstruct it from those samples.

An **unconditioned simulation** generates a random field that follows a given variogram (spatial correlation) and mean, but is not tied to any data. We define a mean value, variance and correlation length per lithology:

In [ ]:
# Property settings per lithology: mean, variance and correlation length (m)
lith_properties = {
    'rock3':    {'mean': 25, 'var': 9, 'len_scale': 300},
    'rock2':    {'mean': 15, 'var': 4, 'len_scale': 500},
    'rock1':    {'mean': 8,  'var': 2, 'len_scale': 200},
    'basement': {'mean': 3,  'var': 1, 'len_scale': 400},
}

def lithology_of(domain_key):
    """Lithology name of a domain, e.g. (2, 0) -> 'rock3'"""
    return domain_names[domain_key].split(' (')[0]

def variogram_of(domain_key):
    """GSTools variogram model for the lithology of a domain"""
    props = lith_properties[lithology_of(domain_key)]
    return gs.Exponential(dim=3, var=props['var'], len_scale=props['len_scale'])

In [ ]:
# Plot the variograms of all lithologies
fig, ax = plt.subplots(figsize=(7, 4))
for lith, props in lith_properties.items():
    gs.Exponential(dim=3, var=props['var'], len_scale=props['len_scale']).plot(
        'variogram', x_max=1500, ax=ax, label=lith)
ax.set_xlabel('Distance (m)')
ax.set_ylabel('Semivariance')
ax.legend()
plt.show()

In [ ]:
# One simulation config per domain (unconditioned: no data needed)
truth_configs = {
    key: SimulationDomainConfig(
        model=variogram_of(key),
        conditioned=False,
        seed=20260 + i,  # different seed per domain, fixed for reproducibility
        srf_kwargs={'mean': lith_properties[lithology_of(key)]['mean']},
    )
    for i, key in enumerate(domain_keys)
}

# Run the simulation (no conditioning data, so we pass an empty data set)
no_data = ConditioningData(x=[], y=[], z=[], values=[])
true_field = run_simulation(geo_model, no_data, truth_configs)

In [ ]:
# 3D view of the "true" property field
plot_property_field(geo_model, true_field, cmap='viridis')

### 4) Conditioning data: virtual boreholes

In a real project, property values come from measurements, e.g. along boreholes. Here we place six vertical boreholes and read the "true" field every 20 m.

`ConditioningData` stores the samples (x, y, z, value). `assign_domains` then assigns every sample to the domain it lies in.

In [ ]:
# Borehole locations (x, y) and sample depths
borehole_xy = [(300, 250), (700, 750), (1100, 400), (1500, 600), (1900, 300), (2200, 800)]
sample_z = np.arange(10, 1000, 20.0)

x = np.concatenate([np.full_like(sample_z, bx) for bx, _ in borehole_xy])
y = np.concatenate([np.full_like(sample_z, by) for _, by in borehole_xy])
z = np.tile(sample_z, len(borehole_xy))

# Read the "true" value of the grid cell closest to each sample
cell_centers = geo_model.grid.regular_grid.values
_, nearest_cell = cKDTree(cell_centers).query(np.column_stack([x, y, z]))
values = true_field.values.ravel()[nearest_cell]

# Store as conditioning data and assign samples to domains
boreholes = ConditioningData(x=x, y=y, z=z, values=values)
boreholes.assign_domains(geo_model, lith_array, fault_array)

for key in domain_keys:
    print(f"{domain_names[key]:28} {len(boreholes.for_domain(key)):4d} samples")

In [ ]:
# 3D view of the samples together with the model surfaces
plot_conditioning_data(geo_model, boreholes, point_size=8)

### 5) Kriging

Kriging estimates the property in every cell from the samples in the same domain. It returns the estimated value and its **variance** (estimation uncertainty): low close to the boreholes, high far away from them.

**Merging domains:** `rock2` in fault block 1 only has a few samples. Since it is the same rock unit as in fault block 0, we can process both fault blocks together as one group, so they share their samples. A group is written as a tuple of domain keys.

**Variograms:** here we know the true variograms, because we created the data ourselves. In a real project you would estimate them from the data (see *Further options* at the end).

In [ ]:
# rock2 on both sides of the fault, processed together
rock2_domains = tuple(key for key in domain_keys if lithology_of(key) == 'rock2')

kriging_configs = {}
for key in domain_keys:
    if key in rock2_domains:
        continue
    kriging_configs[key] = KrigingDomainConfig(model=variogram_of(key))

kriging_configs[rock2_domains] = KrigingDomainConfig(model=variogram_of(rock2_domains[0]))

# Run kriging (ordinary kriging by default)
kriged_field = run_kriging(geo_model, boreholes, kriging_configs)

In [ ]:
# 3D view of the kriging estimate
plot_property_field(geo_model, kriged_field, cmap='viridis')

In [ ]:
# 3D view of the kriging variance (wrapped in a PropertyField so we can use the same plot function)
variance_field = PropertyField(values=kriged_field.variance, variance=kriged_field.variance,
                               domain_keys=kriged_field.domain_keys)
plot_property_field(geo_model, variance_field, cmap='magma')

Comparing truth, kriging estimate and variance in a 2D section (y = 500 m). The dashed lines mark the x positions of the boreholes:

In [ ]:
def plot_sections(fields, titles, cmaps=None, y_index=25):
    """Plot vertical x-z sections of several full-grid arrays side by side"""
    cmaps = cmaps or ['viridis'] * len(fields)
    fig, axes = plt.subplots(1, len(fields), figsize=(5 * len(fields), 3), constrained_layout=True)
    for ax, field, title, cmap in zip(np.atleast_1d(axes), fields, titles, cmaps):
        im = ax.imshow(field[:, y_index, :].T, origin='lower', extent=[0, 2500, 0, 1000], cmap=cmap)
        for bx, _ in borehole_xy:
            ax.axvline(bx, color='white', linestyle='--', linewidth=0.8)
        ax.set_title(title)
        ax.set_xlabel('x (m)')
        ax.set_ylabel('z (m)')
        fig.colorbar(im, ax=ax, shrink=0.8)
    plt.show()

vmin, vmax = np.nanmin(true_field.values), np.nanmax(true_field.values)
plot_sections([true_field.values, kriged_field.values, kriged_field.variance],
              ['"True" field', 'Kriging estimate', 'Kriging variance'],
              cmaps=['viridis', 'viridis', 'magma'])

rmse = np.sqrt(np.nanmean((kriged_field.values - true_field.values) ** 2))
print(f"RMSE kriging vs. truth: {rmse:.2f}")

### 6) Conditioned simulation

Kriging gives the single "best" estimate, which is smoother than reality. A **conditioned simulation** instead generates realistic random fields that honor the borehole data exactly and follow the variogram. Each realization is different. Together they show the range of possible property distributions.

Each realization takes 10-20 seconds at full resolution.

In [ ]:
n_realizations = 3
realizations = []

for r in range(n_realizations):
    simulation_configs = {
        group: SimulationDomainConfig(model=config.model, seed=1000 * r + i)
        for i, (group, config) in enumerate(kriging_configs.items())
    }
    realizations.append(run_simulation(geo_model, boreholes, simulation_configs))

In [ ]:
# Compare realizations in the same 2D section (same color range as the true field)
fig, axes = plt.subplots(1, n_realizations + 1, figsize=(5 * (n_realizations + 1), 3), constrained_layout=True)
for ax, field, title in zip(axes, [true_field] + realizations,
                            ['"True" field'] + [f'Realization {r + 1}' for r in range(n_realizations)]):
    im = ax.imshow(field.values[:, 25, :].T, origin='lower', extent=[0, 2500, 0, 1000],
                   cmap='viridis', vmin=vmin, vmax=vmax)
    for bx, _ in borehole_xy:
        ax.axvline(bx, color='white', linestyle='--', linewidth=0.8)
    ax.set_title(title)
fig.colorbar(im, ax=axes, shrink=0.8)
plt.show()

In [ ]:
# 3D view of the first realization
plot_property_field(geo_model, realizations[0], cmap='viridis')

### 7) Interactive exploration

`plot_property_field_interactive` adds a checkbox and min/max sliders for every domain, e.g. to show only the high values in one unit. Passing the configs shows merged groups (like `rock2`) as one row.

In [ ]:
plot_property_field_interactive(geo_model, kriged_field, domain_configs=kriging_configs, cmap='viridis')

**Further options:**
- **Estimating variograms from data:** `gs.vario_estimate` computes an experimental variogram from the samples of a domain (`boreholes.for_domain(key)`), and `model.fit_variogram` fits a model to it. With only a few boreholes this is difficult: there are many sample pairs along each borehole, but few between boreholes.
- **Kriging method:** `KrigingDomainConfig(model, krige_class=gs.krige.Simple, krige_kwargs={'mean': 10})` uses simple kriging, `gs.krige.Universal` adds a trend.
- **Local neighborhood:** by default all samples of a domain are used. `Neighborhood(mode='n_closest', n=20)` or `Neighborhood(mode='range', radius=...)` (from `gempy_plugins.property_estimation.neighborhood`) uses only nearby samples. This solves one kriging system per cell, so it is much slower, especially at full resolution.
- **Unconditioned and conditioned domains can be mixed** in one `run_simulation` call, e.g. for units without any data.